In [1]:
##############
## INITIATE ##
##############

## Imports
import torch
import torch.nn as nn
import torch.nn.functional as F

#
###

In [2]:
#####################
## PATCH EMBEDDING ##
#####################

class PatchEmbedding(nn.Module):
    
    def __init__(self, in_channels, patch_size, embed_dim, img_size):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # [B, embed_dim, H/patch_size, W/patch_size]
        x = x.flatten(2)  # [B, embed_dim, num_patches]
        x = x.transpose(1, 2)  # [B, num_patches, embed_dim]
        return x

## Fake input
x = torch.randn((4, 128, 64, 64))  # (batch_size, num_channels, height, width)
print(x.size())

## Patch embedding
patch_embed = PatchEmbedding(in_channels=128, patch_size=4, embed_dim=16, img_size=64)
x = patch_embed(x)
print(x.size())

#
###

torch.Size([4, 128, 64, 64])
torch.Size([4, 256, 16])


In [3]:
#########################
## POSITIONAL ENCODING ##
#########################

## Positional embedding vector
pos_embed = nn.Parameter(torch.zeros(1, patch_embed.num_patches, 16))
print(pos_embed.shape)
print(pos_embed[0,0])

## Apply positional embedding vector
x = x + pos_embed
print(x.shape) # No change in size

## Alternatively
pos_embed = torch.zeros(1, patch_embed.num_patches, 16) + torch.arange(0, patch_embed.num_patches)[None, :, None]
print(pos_embed.shape)

#
###

torch.Size([1, 256, 16])
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       grad_fn=<SelectBackward0>)
torch.Size([4, 256, 16])
torch.Size([1, 256, 16])


In [4]:
#########################
## MULTIHEAD ATTENTION ##
#########################

## Initialize Multihead Attention with the number of channels as the embedding size
attention = nn.MultiheadAttention(embed_dim=16, num_heads=2, dropout=0.1)

## Multihead attention
att, attn_weights = attention(x, x, x)  # Query, Key, Value are the same
print(attn_weights.shape) 
print(x.shape)  # No change in size

## Apply attention
x = x + att
print(x.shape)

#
###

torch.Size([256, 4, 4])
torch.Size([4, 256, 16])
torch.Size([4, 256, 16])


In [5]:
#########
## MLP ##
#########

## Hyperparameters
embed_dim = 16
mlp_dim = 256
dropout = 0.1

## Define mlp
mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

## Output
x = x + mlp(x)
print(x.shape)

#
###

torch.Size([4, 256, 16])


In [6]:
##############
## INITIATE ##
##############

import torch
import torch.nn as nn

#
###

#############
## CLASSES ##
#############

class PatchEmbedding(nn.Module):
    
    def __init__(self, in_channels, patch_size, embed_dim, img_size):
        super().__init__()
        self.patch_size = patch_size
        self.num_patches = (img_size // patch_size) ** 2
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.proj(x)  # (B, E, H/patch_size, W/patch_size); P = H/patch_size * W/patch_size
        x = x.flatten(2)  # (B, E, P)
        x = x.transpose(1, 2)  # (B, P, E)
        return x

class TransformerEncoderLayer(nn.Module):
    
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        attn_output, _ = self.attn(x, x, x)
        x = x + attn_output
        x = self.norm1(x)
        mlp_output = self.mlp(x)
        x = x + mlp_output
        x = self.norm2(x)
        return x

class SimpleVisionTransformer(nn.Module):
    
    def __init__(self, img_size=64, patch_size=8, in_channels=3, embed_dim=128, depth=4, num_heads=8, mlp_dim=256, dropout=0.1):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, patch_size, embed_dim, img_size)
        self.pos_embed = torch.zeros(1, self.patch_embed.num_patches, embed_dim) + torch.arange(0, self.patch_embed.num_patches)[None, :, None]
        # self.pos_embed = nn.Parameter(torch.zeros(1, self.patch_embed.num_patches, embed_dim)) # For learnable positional encoding
        self.pos_drop = nn.Dropout(dropout)
        self.transformer = nn.Sequential(*[TransformerEncoderLayer(embed_dim, num_heads, mlp_dim, dropout) for _ in range(depth)])

    def forward(self, x):
        x = self.patch_embed(x) # (B, C, H, W) --> (B, P, E)
        x = x + self.pos_embed
        x = self.pos_drop(x)
        x = x.transpose(0, 1)  # (B, P, E) --> (P, B, E) as this is the convention for multihead attention
        x = self.transformer(x)
        x = x.transpose(0, 1)  # (P, B, E) --> (B, P, E)
        return x

#
###

In [7]:
################
## ALTOGETHER ##
################

## Fake input
x = torch.randn((4, 3, 64, 64))  # (batch_size, num_channels, height, width)
print(x.size())  # Should print (4, 128, 64, 64)

## Instantiate the ViT
model = SimpleVisionTransformer()

## Apply to image batch
yhat = model(x)
print(yhat.size())

#
###

torch.Size([4, 3, 64, 64])
torch.Size([4, 64, 128])


In [12]:
##############
## INITIATE ##
##############

import torch
import torch.nn as nn

#
###

#############
## CLASSES ##
#############

class TransformerEncoderLayer(nn.Module):
    
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        attention, _ = self.attention(x, x, x)
        print(_.size())
        x = x + attention
        x = self.norm1(x)
        mlp_output = self.mlp(x)
        x = x + mlp_output
        x = self.norm2(x)
        return x

class SegmentTransformer(nn.Module):
    
    def __init__(self, input_size=64, embed_dim=128, depth=1, num_heads=8, mlp_dim=256, dropout=0.1):
        super().__init__()
        self.num_patches = input_size * input_size
        self.pos_embed = torch.zeros(1, embed_dim, self.num_patches) + torch.arange(0, self.num_patches)[None, None, :]
        # self.pos_embed = nn.Parameter(torch.zeros(1, embed_dim, self.num_patches)) # For learnable positional encoding
        self.pos_drop = nn.Dropout(dropout)
        self.transformer = nn.Sequential(*[TransformerEncoderLayer(embed_dim, num_heads, mlp_dim, dropout) for _ in range(depth)])

    def forward(self, x): # (B, C, H, W)
        B, C, H, W = x.size()
        print(x.size())
        x = x.reshape(B, C, H * W) # (B, C, H, W) --> (B, C, P)
        print(x.size())
        x = x + self.pos_embed
        x = self.pos_drop(x)
        x = torch.permute(x, (2, 0, 1))  # (B, C, P) --> (P, B, C) as this is the convention for multihead attention
        print(x.size())
        x = self.transformer(x)
        print(x.size())
        x = torch.permute(x, (1, 2, 0))  # (P, B, C) --> (B, C, P)
        print(x.size())
        x = x.reshape(B, C, H, W) # (B, C, P) --> (B, C, H, W) 
        print(x.size())
        return x

#
###

In [13]:
###############################
## CHECK SEGMENT TRANSFORMER ##
###############################

## Fake input
x = torch.randn((4, 128, 64, 64))  # (batch_size, num_channels, height, width)
print(x.size())  # Should print (4, 128, 64, 64)

## Instantiate the ViT
model = SegmentTransformer(input_size=64, embed_dim=128)

## Apply to image batch
yhat = model(x)
print(yhat.size())

#
###

torch.Size([4, 128, 64, 64])
torch.Size([4, 128, 64, 64])
torch.Size([4, 128, 4096])
torch.Size([4096, 4, 128])
torch.Size([4, 4096, 4096])
torch.Size([4096, 4, 128])
torch.Size([4, 128, 4096])
torch.Size([4, 128, 64, 64])
torch.Size([4, 128, 64, 64])


In [19]:
############################
## f_model_transformer.py ##
############################

## Goal: Define a UNet-Transfomer model for semantic segmentation.
## Author: Willem Bonnaffe (w.bonnaffe@gmail.com) 

##############
## INITIATE ##
##############

import torch
import torch.nn as nn
import torch.nn.functional as F

#
###

#############
## CLASSES ##
#############

## DoubleConv
## Goal:
## Perform two consecutive convolutional operations with Batch Normalization and ReLU activation.
## Inputs:
## in_channels (int): Number of input channels.
## out_channels (int): Number of output channels.
## Outputs:
## x (torch.Tensor): Transformed tensor after double convolution.
class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels, track_running_stats=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.BatchNorm2d(out_channels, track_running_stats=False),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        x = self.conv(x)
        return x

## Down
## Goal:
## Perform downsampling using max pooling followed by a double convolution.
## Inputs:
## in_channels (int): Number of input channels.
## out_channels (int): Number of output channels.
## Outputs:
## x (torch.Tensor): Downsampled and transformed tensor.
class Down(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.pool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels)
        )

    def forward(self, x):
        x = self.pool_conv(x)
        return x

## Up
## Goal:
## Perform upsampling followed by a double convolution, with optional bilinear interpolation.
## Inputs:
## in_channels (int): Number of input channels.
## out_channels (int): Number of output channels.
## bilinear (bool): Whether to use bilinear interpolation for upsampling.
## Outputs:
## x (torch.Tensor): Upsampled and transformed tensor.
class Up(nn.Module):

    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()

        ## Upsampling layer
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        else:
            self.up = nn.ConvTranspose2d(in_channels // 2, in_channels // 2, kernel_size=2, stride=2)

        ## Double conv layer
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        x = self.conv(x)
        return x

## OutConv
## Goal:
## Perform a final 1x1 convolution to produce the desired number of output channels.
## Inputs:
## in_channels (int): Number of input channels.
## out_channels (int): Number of output channels.
## Outputs:
## x (torch.Tensor): Output tensor with the desired number of channels.    
class OutConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        x = self.conv(x)
        return x

## TransformerEncoderLayer
## Goal:
## Define a single encoder layer for a Transformer model with self-attention and feed-forward network.
## Inputs:
## embed_dim (int): Dimension of the embedding.
## num_heads (int): Number of attention heads.
## mlp_dim (int): Dimension of the feed-forward network.
## dropout (float): Dropout rate.
## Outputs:
## x (torch.Tensor): Transformed tensor after attention and MLP operations.
class TransformerEncoderLayer(nn.Module):
    
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attention = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )

    def forward(self, x):
        attention, _ = self.attention(x, x, x)
        x = x + attention
        x = self.norm1(x)
        mlp_output = self.mlp(x)
        x = x + mlp_output
        x = self.norm2(x)
        return x

## SegmentTransformer
## Goal:
## Implement a Transformer model for image segmentation, including positional encoding and Transformer layers.
## Inputs:
## input_size (int): Size of the input image.
## embed_dim (int): Dimension of the embedding.
## depth (int): Number of Transformer layers.
## num_heads (int): Number of attention heads.
## mlp_dim (int): Dimension of the feed-forward network.
## dropout (float): Dropout rate.
## Outputs:
## x (torch.Tensor): Transformed tensor after the Transformer layers.
class SegmentTransformer(nn.Module):
    
    def __init__(self, input_size=64, embed_dim=128, depth=1, num_heads=8, mlp_dim=256, dropout=0.1):
        super().__init__()
        self.num_patches = input_size * input_size
        # self.pos_embed = torch.zeros(1, embed_dim, self.num_patches) + torch.arange(0, self.num_patches)[None, None, :]
        self.pos_embed = nn.Parameter(torch.zeros(1, embed_dim, self.num_patches)) # For learnable positional encoding
        self.pos_drop = nn.Dropout(dropout)
        self.transformer = nn.Sequential(*[TransformerEncoderLayer(embed_dim, num_heads, mlp_dim, dropout) for _ in range(depth)])

    def forward(self, x): # (B, C, H, W)
        print(x.size())
        B, C, H, W = x.size()
        x = x.reshape(B, C, H * W) # (B, C, H, W) --> (B, C, P)
        print(x.size())
        x = x + self.pos_embed
        x = self.pos_drop(x)
        x = torch.permute(x, (2, 0, 1))  # (B, C, P) --> (P, B, C) as this is the convention for multihead attention
        print(x.size())
        x = self.transformer(x)
        print(x.size())
        x = torch.permute(x, (1, 2, 0))  # (P, B, C) --> (B, C, P)
        print(x.size())
        x = x.reshape(B, C, H, W) # (B, C, P) --> (B, C, H, W) 
        print(x.size())
        return x

## UNetTransformer
## Goal:
## Combine a U-Net architecture with a Transformer encoder for enhanced semantic segmentation performance.
## Inputs:
## n_channels (int): Number of input channels.
## n_classes (int): Number of output classes.
## bilinear (bool): Whether to use bilinear upsampling.
## input_size (int): Size of the input images.
## embed_dim (int): Dimension of the embedding.
## depth (int): Number of Transformer layers.
## num_heads (int): Number of attention heads.
## mlp_dim (int): Dimension of the feed-forward network.
## dropout (float): Dropout rate.
## Outputs:
## logits (torch.Tensor): Output tensor with logits for each class.
class UNetTransformer(nn.Module):

    def __init__(self, n_channels=3, n_classes=1, bilinear=True, input_size=64, embed_dim=128, depth=2, num_heads=8, mlp_dim=256, dropout=0.1):
        super(UNetTransformer, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        self.inc = DoubleConv(n_channels, 16)
        self.down1 = Down(16, 32) # H/2
        self.down2 = Down(32, 64) # H/4
        self.down3 = Down(64, 128) # H/8
        self.down4 = Down(128, 128) # H/16 (e.g. 1024 -> 512 -> 256 -> 128 -> 64)
        self.transf = SegmentTransformer(input_size, embed_dim, depth, num_heads, mlp_dim, dropout)
        self.up1 = Up(256, 64)
        self.up2 = Up(128, 32)
        self.up3 = Up(64, 16)
        self.up4 = Up(32, 16)
        self.outc = OutConv(16, n_classes)        

    def forward(self, x):
        print(x.size())
        print("*in convolution*")
        x1 = self.inc(x)
        print(x1.size())
        print("*down convolution*")
        x2 = self.down1(x1)
        print(x2.size())
        x3 = self.down2(x2)
        print(x3.size())
        x4 = self.down3(x3)
        print(x4.size())
        x5 = self.down4(x4)
        print(x5.size())
        print("*transformer*")
        x5 = self.transf(x5) # Transformer
        print(x5.size())
        print("*up convolution*")
        x = self.up1(x5, x4)
        print(x.size())
        x = self.up2(x, x3)
        print(x.size())
        x = self.up3(x, x2)
        print(x.size())
        x = self.up4(x, x1)
        print(x.size())
        print("*out convolution*")
        logits = self.outc(x)
        print(x.size())
        return logits

#
###

In [20]:
############################
## CHECK UNET TRANSFORMER ##
############################

## Fake input
x = torch.randn((1, 3, 1024, 1024))  # (batch_size, num_channels, height, width)
print(x.size())

## Instantiate the ViT
model = UNetTransformer()

## Apply to image batch
yhat = model(x)
print(yhat.size())

#
###

torch.Size([1, 3, 1024, 1024])
torch.Size([1, 3, 1024, 1024])
*in convolution*
torch.Size([1, 16, 1024, 1024])
*down convolution*
torch.Size([1, 32, 512, 512])
torch.Size([1, 64, 256, 256])
torch.Size([1, 128, 128, 128])
torch.Size([1, 128, 64, 64])
*transformer*
torch.Size([1, 128, 64, 64])
torch.Size([1, 128, 4096])
torch.Size([4096, 1, 128])
torch.Size([4096, 1, 128])
torch.Size([1, 128, 4096])
torch.Size([1, 128, 64, 64])
torch.Size([1, 128, 64, 64])
*up convolution*
torch.Size([1, 64, 128, 128])
torch.Size([1, 32, 256, 256])
torch.Size([1, 16, 512, 512])
torch.Size([1, 16, 1024, 1024])
*out convolution*
torch.Size([1, 16, 1024, 1024])
torch.Size([1, 1, 1024, 1024])


In [ ]:
###############################
## CHECK MULTIHEAD ATTENTION ##
###############################

## Fake input
x = torch.randn((4, 128, 64, 64))  # (batch_size, num_channels, height, width)
print(x.shape)  # Should print (4, 128, 64, 64)

##
batch_size, num_channels, height, width = x.shape

## Reformat x for MultiheadAttention
x = x.reshape(batch_size, num_channels, height * width)  # Reshape to (batch_size, num_channels, height * width)
x = x.permute(2, 0, 1)  # Correct transpose to (height * width, batch_size, num_channels)
print(x.shape)  # Should print (height * width, batch_size, num_channels)

## Initialize Multihead Attention with the number of channels as the embedding size
attention = nn.MultiheadAttention(embed_dim=num_channels, num_heads=2, dropout=0.1)

## Multihead attention
x, attn_weights = attention(x, x, x)  # Query, Key, Value are the same
print(attn_weights.shape) 
print(x.shape)  # Should print (height * width, batch_size, num_channels)

#
###